In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/NYC_Taxi_ABT.csv')
initial_rows = len(df)
print(f"Initial ABT rows loaded: {initial_rows:,}")

Initial ABT rows loaded: 3,946,695


In [3]:
# 1. Remove negative fees, taxes, and fare adjustments
fee_cols = [
    'cbd_congestion_fee', 'congestion_surcharge', 'Airport_fee', 
    'mta_tax', 'extra', 'improvement_surcharge', 'tolls_amount', 
    'tip_amount', 'total_fees', 'fee_impact_pct'
]
df = df[(df[fee_cols] >= 0).all(axis=1)].copy()
after_fees = len(df)
print(f"Rows after removing negative fees/taxes: {after_fees:,} (Dropped {initial_rows - after_fees:,})")

# 2. Ensure strictly positive fare and net revenue amounts
df = df[(df['fare_amount'] > 0) & (df['total_amount'] > 0) & (df['net_revenue'] > 0)].copy()
after_revenue = len(df)
print(f"Rows after enforcing positive revenue:   {after_revenue:,} (Dropped {after_fees - after_revenue:,})")

# 3. Remove zero-passenger trips and invalid TLC RatecodeIDs (valid codes are 1 to 6)
df = df[(df['passenger_count'] > 0) & (df['RatecodeID'].between(1, 6))].copy()
after_validity = len(df)
print(f"Rows after removing invalid trips/codes: {after_validity:,} (Dropped {after_revenue - after_validity:,})")

# 4. Filter extreme target outliers (capping net_revenue_per_hour at $500/hr)
df = df[df['net_revenue_per_hour'] <= 500].copy()
final_rows = len(df)
print(f"Rows after capping target outliers:      {final_rows:,} (Dropped {after_validity - final_rows:,})")

print(f"\nTotal rows removed: {initial_rows - final_rows:,} ({((initial_rows - final_rows) / initial_rows) * 100:.2f}%)")

Rows after removing negative fees/taxes: 3,946,620 (Dropped 75)
Rows after enforcing positive revenue:   3,946,620 (Dropped 0)
Rows after removing invalid trips/codes: 3,857,221 (Dropped 89,399)
Rows after capping target outliers:      3,852,830 (Dropped 4,391)

Total rows removed: 93,865 (2.38%)


In [4]:
print("----- FINAL CHECK -----")
print(f"Final Row Count:       {len(df):,}")
print(f"Total Missing Values:  {df.isnull().sum().sum()}")
print(f"Min Total Fees:        ${df['total_fees'].min():.2f}")
print(f"Min Fee Impact %:      {df['fee_impact_pct'].min():.2f}%")
print(f"Min Passenger Count:   {df['passenger_count'].min()}")
print(f"Max RatecodeID:        {df['RatecodeID'].max()}")
print(f"Min Speed (mph):       {df['speed_mph'].min():.2f}")
print(f"Max Net Revenue/Hour:  ${df['net_revenue_per_hour'].max():.2f}/hr")

----- FINAL CHECK -----
Final Row Count:       3,852,830
Total Missing Values:  0
Min Total Fees:        $0.00
Min Fee Impact %:      0.00%
Min Passenger Count:   1
Max RatecodeID:        6
Min Speed (mph):       1.00
Max Net Revenue/Hour:  $500.00/hr


In [5]:
output_path = '../data/Final_NYC_Taxi_Cleaned.csv'
df.to_csv(output_path, index=False)